In [ ]:
import pandas as pd

df_lrp_95p_spike = pd.read_csv(
    r"C:/Users/satavans/Downloads/shapelet_files/lrp_results/metrics_top10.csv"
)
df_lrp_95p_spike["method"] = "ROCKET-LRP"


df_mfdr_95p_spike = pd.read_csv(
    r"C:/Users/satavans/Downloads/shapelet_files/results/metrics_top10_0.1.csv"
)
df_mfdr_95p_spike["method"] = "FDF-ROCKET (0.1)"


df_mfdr_95p_spike1 = pd.read_csv(
    r"C:/Users/satavans/Downloads/shapelet_files/results/metrics_top10_0.01.csv"
)
df_mfdr_95p_spike1["method"] = "FDF-ROCKET (0.01)"


df_mfdr_95p_spike2 = pd.read_csv(
    r"C:/Users/satavans/Downloads/shapelet_files/results/metrics_top10_0.001.csv"
)
df_mfdr_95p_spike2["method"] = "FDF-ROCKET (0.001)"


df_shap_95p_spike = pd.read_csv(
    r"C:/Users/satavans/Downloads/shapelet_files/shap_results/metrics_topk.csv"
)
df_shap_95p_spike["method"] = "Kernel SHAP"


df_lime_95p_spike = pd.read_csv(
    r"C:/Users/satavans/Downloads/shapelet_files/lime_results/metrics_top10.csv"
)
df_lime_95p_spike["method"] = "LIME"


df_rand_95p_spike = pd.read_csv(
    r"C:/Users/satavans/Downloads/shapelet_files/random_results/metrics_topk.csv"
)
df_rand_95p_spike["method"] = "Random"

In [ ]:
import pandas as pd
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests
import itertools


# combine
dfs = [
    df_lrp_95p_spike,
    df_mfdr_95p_spike,
    df_mfdr_95p_spike1,
    df_mfdr_95p_spike2,
    df_shap_95p_spike,
    df_lime_95p_spike,
    df_rand_95p_spike
]


# Add iteration ID
for df in dfs:
    df["iteration"] = range(len(df))


recall_df = pd.concat(
    [df[["iteration","method","recall"]] for df in dfs]
)


# Make matrix
recall_matrix = recall_df.pivot(
    index="iteration",
    columns="method",
    values="recall"
).dropna()


print(recall_matrix.shape)

In [ ]:
results = []

methods = recall_matrix.columns


for m1, m2 in itertools.combinations(methods, 2):

    stat, p = ttest_rel(
        recall_matrix[m1],
        recall_matrix[m2]
    )

    mean_diff = (
        recall_matrix[m1].mean()
        -
        recall_matrix[m2].mean()
    )

    results.append([
        m1,
        m2,
        mean_diff,
        p
    ])


results_df = pd.DataFrame(
    results,
    columns=[
        "Method 1",
        "Method 2",
        "Mean Recall Difference",
        "p-value"
    ]
)


# Holm correction
reject, p_corrected, _, _ = multipletests(
    results_df["p-value"],
    method="holm"
)


results_df["p-adjusted"] = p_corrected
results_df["Significant"] = reject


results_df.sort_values(
    "p-adjusted"
)

In [ ]:
from autorank import autorank, plot_stats


result = autorank(
    recall_matrix,
    alpha=0.05,
    order="descending"
)


plot_stats(result)

In [ ]:
from autorank import plot_stats
import matplotlib.pyplot as plt

plot_stats(result)

fig = plt.gcf()
ax = plt.gca()

# Remove only CD text
for text in ax.texts[:]:
    if "CD" in text.get_text():
        text.remove()

# Inspect lines first
for i, line in enumerate(ax.lines):
    print(i, line.get_xdata(), line.get_ydata())

plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import sys
import numpy as np
import pandas as pd

sys.path.insert(0, r"C:\Users\satavans\Downloads\rocket-lrp")
from rocket_lrp.rocket import RocketTransform
from rocket_lrp.lrp import RocketLRPExplainer
from rocket_lrp.explainers import LinearSHAPExplainer
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
import scipy
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import sktime
import sklearn
from scipy.io import loadmat
from sktime.transformations.panel.rocket import Rocket
# import timesynth as ts
import random
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

def filter_latent_importances(R_h, alpha_percent=30.0):
    """Zero the lowest alpha_percent of absolute latent importances per sample."""
    R_h = np.asarray(R_h, dtype=np.float64)
    if alpha_percent <= 0:
        return R_h
    if R_h.ndim == 1:
        R_h = R_h[None, :]

    out = R_h.copy()
    mag = np.abs(out)
    thr = np.percentile(mag, alpha_percent, axis=1, keepdims=True)
    out[mag < thr] = 0.0
    return out


def explain_with_paper_lrp(train_X3, test_X, test_X2, y, alpha_percent=30.0, epsilon=1e-6):
    """Run the paper's ROCKET + LinearSHAP + LRP workflow exactly for one split."""

    trf = Rocket(num_kernels=1000, random_state=1234)
    trf.fit(train_X3)

    rocket = RocketTransform.from_sktime(
        trf,
        train_X3
    )

    H_train = trf.transform(train_X3)
    scaler = StandardScaler().fit(H_train)
    H_train_scaled = scaler.transform(H_train)

    classifier = LogisticRegression(
        max_iter=1000,
    ).fit(H_train_scaled, y)

    # Stage 1: paper-style LinearSHAP in the latent ROCKET feature space.
    baseline = H_train_scaled.mean(axis=0)
    stage1 = LinearSHAPExplainer(
        classifier.coef_.ravel(),
        baseline=baseline,
        scaler_mean=scaler.mean_,
        scaler_scale=scaler.scale_,
    )

    H_test = trf.transform(test_X2)
    H_test_scaled = scaler.transform(H_test)
    R_h = stage1.explain(H_test_scaled)

    # Stage 2: paper-style LRP propagation to the original input space.
    lrp = RocketLRPExplainer(rocket, epsilon=epsilon, alpha=alpha_percent)
    R_h = filter_latent_importances(R_h, alpha_percent=alpha_percent)
    R_lrp = lrp.explain(test_X2, R_h)

    return classifier, scaler, trf, R_lrp

i = 0
rocket_features = pd.read_csv('C:/Users/satavans/Downloads/alcohol_application/train_files/driving_split' + str(i) + '.csv')
X_train = np.load("C:/Users/satavans/Downloads/alcohol_application/train_raw/X_train" + str(i) + ".npy")
X_test = np.load("C:/Users/satavans/Downloads/alcohol_application/test_raw/X_test" + str(i) + ".npy")
y_train = np.load("C:/Users/satavans/Downloads/alcohol_application/train_raw/y_train" + str(i) + ".npy")
y_test = np.load("C:/Users/satavans/Downloads/alcohol_application/test_raw/y_test" + str(i) + ".npy")
X_train2 = X_train.copy()

classifier, scaler, rocket, R_lrp = explain_with_paper_lrp(
    X_train,
    X_test,
    X_test,
    y_train,
    alpha_percent=30.0,
    epsilon=1e-6,
)

sample = 0

# Absolute relevance scores
importance_lrp = np.abs(R_lrp[sample, 0, :])

k5 = int(0.05 * len(importance_lrp))
k10 = int(0.10 * len(importance_lrp))
k15 = int(0.15 * len(importance_lrp))

top5 = np.argsort(importance_lrp)[-k5:]
top10 = np.argsort(importance_lrp)[-k10:]
top15 = np.argsort(importance_lrp)[-k15:]

signal = X_test[sample, 0]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
import scipy
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import sktime
import sklearn
from scipy.io import loadmat
from sktime.transformations.panel.rocket import Rocket
# import timesynth as ts
import random
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

n_steps = 3600
sample = 0
i = 0
# get original segment for given sample that is impacted by a kernel
def get_original_segment(sample, impacted_k_weights,
    impacted_k_bias,
    impacted_k_dilation,
    impacted_k_padding,
    impacted_k_lengths):
    sample_data = X_test[sample, 0, :]

    # check for padding and apply to the segment if needed on both left and right sides
    if impacted_k_padding != 0:
        padding_left = ((impacted_k_lengths - 1) * impacted_k_dilation) // 2
        padding_right = ((impacted_k_lengths - 1) * impacted_k_dilation) - padding_left
        padded_segment = np.pad(sample_data, (padding_left, padding_right), mode='constant')
    else:
        padded_segment = sample_data.copy()
        
    conv = np.zeros(n_steps)
    for i in range(n_steps):
        indices = i + np.arange(impacted_k_lengths) * impacted_k_dilation
        if np.all(indices < len(padded_segment)):
            conv[i] = np.sum(padded_segment[indices] * impacted_k_weights)
    return conv + impacted_k_bias

# get injected segment
def get_injected_segment(sample, impacted_k_weights,
    impacted_k_bias,
    impacted_k_dilation,
    impacted_k_padding,
    impacted_k_lengths):
    sample_data = X_train[sample, 0, :]

    # check for padding and apply to the segment if needed on both left and right sides
    if impacted_k_padding != 0:
        padding_left = ((impacted_k_lengths - 1) * impacted_k_dilation) // 2
        padding_right = ((impacted_k_lengths - 1) * impacted_k_dilation) - padding_left
        padded_segment = np.pad(sample_data, (padding_left, padding_right), mode='constant')
    else:
        padded_segment = sample_data.copy()

    conv = np.zeros(n_steps)
    for i in range(n_steps):
        indices = i + np.arange(impacted_k_lengths) * impacted_k_dilation
        if np.all(indices < len(padded_segment)):
            conv[i] = np.sum(padded_segment[indices] * impacted_k_weights)
    return conv + impacted_k_bias


def compute_importance(selected_kernels):
    all_point_diffs = {}
 
    for kernel in selected_kernels:
        kernel_start = np.sum(kernel_lengths[:kernel])
        kernel_end = kernel_start + kernel_lengths[kernel]
 
        impacted_k_weights = kernel_weights[kernel_start:kernel_end]
        impacted_k_bias = kernel_biases[kernel]
        impacted_k_dilation = kernel_dilations[kernel]
        impacted_k_padding = kernel_paddings[kernel]
        impacted_k_length = kernel_lengths[kernel]
 
        baseline = get_original_segment(
            sample,
            impacted_k_weights,
            impacted_k_bias,
            impacted_k_dilation,
            impacted_k_padding,
            impacted_k_length
        )
 
        for idx, value in enumerate(baseline):
            all_point_diffs[idx] = all_point_diffs.get(idx, 0) + abs(value)
 
    importance = np.zeros(X_test.shape[2])
 
    for idx, value in all_point_diffs.items():
        importance[idx] = value
 
    return importance

rocket_features = pd.read_csv('C:/Users/satavans/Downloads/alcohol_application/train_files/driving_split' + str(i) + '.csv')
X_train = np.load("C:/Users/satavans/Downloads/alcohol_application/train_raw/X_train" + str(i) + ".npy")
X_test = np.load("C:/Users/satavans/Downloads/alcohol_application/test_raw/X_test" + str(i) + ".npy")
y_train = np.load("C:/Users/satavans/Downloads/alcohol_application/train_raw/y_train" + str(i) + ".npy")
y_test = np.load("C:/Users/satavans/Downloads/alcohol_application/test_raw/y_test" + str(i) + ".npy")
X_test2 = X_test.copy()

# fit the rocket transformer on sober drive data
trf = Rocket(num_kernels=1000, random_state = 1234)
rocket_features = trf.fit_transform(X_train)
model = LogisticRegression(max_iter = 1000)
scaler = StandardScaler()
rocket_features = scaler.fit_transform(rocket_features)
model.fit(rocket_features, y_train)

X_test_rocket = trf.transform(X_test)
X_test_rocket = scaler.transform(X_test_rocket)
kernel_weights = trf.kernels[0]
kernel_lengths = trf.kernels[1]
kernel_biases = trf.kernels[2]
kernel_dilations = trf.kernels[3]
kernel_paddings = trf.kernels[4]

kernels001 = pd.read_csv(
    f"C:/Users/satavans/Downloads/alcohol_application/kernels_0.01/spike_kernels{i}.csv"
)
 
kernels01 = pd.read_csv(
    f"C:/Users/satavans/Downloads/alcohol_application/kernels_0.1/spike_kernels{i}.csv"
)

kernels0001 = pd.read_csv(
    f"C:/Users/satavans/Downloads/alcohol_application/kernels_0.001/spike_kernels{i}.csv"
)
 
selected001 = kernels001["selected_features"].values
selected0001 = kernels0001["selected_features"].values
selected01 = kernels01["selected_features"].values
 
importance001 = compute_importance(selected001)
importance0001 = compute_importance(selected0001)
importance01 = compute_importance(selected01)
 
k = int(0.05 * len(importance001))

top0001 = np.argsort(importance0001)[-k:] 
top001 = np.argsort(importance001)[-k:]
top01 = np.argsort(importance01)[-k:]
 
signal = X_test[sample, 0]
 
fig, axes = plt.subplots(3, 1, figsize=(16, 8), sharex=True)
 
# ----- Top 5% -----
axes[0].plot(signal, color="steelblue", lw=1.5)
axes[0].scatter(
    top5,
    signal[top5],
    color="red",
    s=20,
    label="Top 5%"
)
axes[0].set_title("ROCKET-LRP Top 5% Relevance")
axes[0].legend()
 
# -------- kernel_0.1 --------
axes[1].plot(signal, color="steelblue", lw=1.5)
axes[1].scatter(
    top01,
    signal[top01],
    color="darkorange",
    s=20,
    label="Top 5%"
)
axes[1].set_title("Selected Kernels (0.1)")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Signal")
axes[1].legend()

# -------- kernel_0.001 --------
axes[2].plot(signal, color="steelblue", lw=1.5)
axes[2].scatter(
    top0001,
    signal[top0001],
    color="red",
    s=20,
    label="Top 5%"
)
axes[2].set_title("Selected Kernels (0.001)")
axes[2].set_ylabel("Signal")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# Extract the top 5% most relevant time points for every test sample
# ------------------------------------------------------------------

# Absolute relevance scores
importance_all = np.abs(R_lrp[:, 0, :])  # Shape: (n_samples, n_timesteps)

n_samples, n_timesteps = importance_all.shape
k5 = int(np.ceil(0.05 * n_timesteps))

top5_indices = []
top5_scores = []
rows = []

for sample in range(n_samples):
    scores = importance_all[sample]

    # Indices of the top 5% relevance scores
    idx = np.argsort(scores)[-k5:]

    # Sort from highest relevance to lowest (optional)
    idx = idx[np.argsort(scores[idx])[::-1]]

    top5_indices.append(idx)
    top5_scores.append(scores[idx])

    # Store in long-format table
    for t in idx:
        rows.append({
            "sample": sample,
            "timepoint": t,
            "relevance": scores[t]
        })

top5_indices = np.array(top5_indices)   # Shape: (n_samples, k5)
top5_scores = np.array(top5_scores)     # Shape: (n_samples, k5)

# DataFrame containing every important point
df_top5 = pd.DataFrame(rows)

print(df_top5.head())
print(f"\nTotal important points: {len(df_top5)}")
print(f"Points per sample: {k5}")

# ------------------------------------------------------------------
# Plot 1: Distribution of important time points
# ------------------------------------------------------------------

plt.figure(figsize=(10, 4))
plt.hist(df_top5["timepoint"], bins=100, edgecolor="black")
plt.xlabel("Time Point")
plt.ylabel("Frequency")
plt.title("Distribution of Top 5% Important Time Points")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# Plot 2: Distribution of relevance scores
# ------------------------------------------------------------------

plt.figure(figsize=(10, 4))
plt.hist(df_top5["relevance"], bins=100, edgecolor="black")
plt.xlabel("Absolute Relevance")
plt.ylabel("Frequency")
plt.title("Distribution of Top 5% Relevance Scores")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# Plot 3: Mean relevance at each time point
# ------------------------------------------------------------------

mean_relevance = (
    df_top5.groupby("timepoint")["relevance"]
    .mean()
    .reindex(range(n_timesteps), fill_value=0)
)

plt.figure(figsize=(12, 4))
plt.plot(mean_relevance.index, mean_relevance.values)
plt.xlabel("Time Point")
plt.ylabel("Mean Absolute Relevance")
plt.title("Mean Relevance of Top 5% Important Points Across Test Set")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------
# Optional: Save results
# ------------------------------------------------------------------

# df_top5.to_csv("top5_lrp_points.csv", index=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Collect raw signal values at the top 5% most relevant points
# ---------------------------------------------------------

importance_all = np.abs(R_lrp[:, 0, :])   # (n_samples, n_timesteps)
signals = X_test[:, 0, :]                 # (n_samples, n_timesteps)

n_samples, n_timesteps = importance_all.shape
k5 = int(np.ceil(0.05 * n_timesteps))

important_values1 = []
important_scores1 = []
important_indices1 = []

for sample in range(n_samples):
    scores = importance_all[sample]

    # Top 5% most relevant time points
    idx = np.argsort(scores)[-k5:]

    # Store information
    important_indices1.extend(idx)
    important_scores1.extend(scores[idx])
    important_values1.extend(signals[sample, idx])

important_values1 = np.array(important_values1)
important_scores1 = np.array(important_scores1)
important_indices1 = np.array(important_indices1)

print(f"Collected {len(important_values1)} important points.")

# ---------------------------------------------------------
# Histogram of raw feature values
# ---------------------------------------------------------

plt.figure(figsize=(8,5))
plt.hist(important_values1, bins=50, edgecolor='black')
plt.xlabel("Raw Time Series Feature Value")
plt.ylabel("Frequency")
plt.title("Distribution of Raw Values at Top 5% LRP-Relevant Time Points")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -------------------------------------------------------
# Distribution of raw signal values for kernel_0.1
# -------------------------------------------------------

selected01 = kernels01["selected_features"].values

important_values = []
important_scores = []
important_indices = []

k = int(np.ceil(0.05 * X_test.shape[2]))

for sample in range(X_test.shape[0]):

    # Compute importance for this sample
    importance = compute_importance(selected01)

    # Top 5% indices
    top_idx = np.argsort(importance)[-k:]

    # Raw signal values at those locations
    signal = X_test[sample, 0]

    important_values.extend(signal[top_idx])
    important_scores.extend(importance[top_idx])
    important_indices.extend(top_idx)

important_values = np.array(important_values)
important_scores = np.array(important_scores)
important_indices = np.array(important_indices)

print(f"Collected {len(important_values)} important points.")

# -------------------------------------------------------
# Histogram of raw feature values
# -------------------------------------------------------

plt.figure(figsize=(8,5))
plt.hist(important_values, bins=50, edgecolor="black")
plt.xlabel("Raw Time Series Feature Value")
plt.ylabel("Frequency")
plt.title("Kernel Method (0.1): Raw Values at Top 5% Important Points")
plt.tight_layout()
plt.show()

# -------------------------------------------------------
# Optional: Raw value vs importance
# -------------------------------------------------------

plt.figure(figsize=(8,5))
plt.scatter(important_values, important_scores, alpha=0.4, s=10)
plt.xlabel("Raw Time Series Feature Value")
plt.ylabel("Importance")
plt.title("Kernel Method (0.1): Raw Value vs Importance")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(important_values, bins=50, edgecolor="black")
plt.xlabel("Raw Time Series Feature Value")
plt.ylabel("Frequency")
plt.title("Kernel Method (0.1): Raw Values at Top 5% Important Points")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,5))
plt.hist(important_values1, bins=50, edgecolor='black')
plt.xlabel("Raw Time Series Feature Value")
plt.ylabel("Frequency")
plt.title("Distribution of Raw Values at Top 5% LRP-Relevant Time Points")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Common bins for both distributions
bins = np.histogram_bin_edges(
    np.concatenate([important_values, important_values1]),
    bins=50
)

plt.figure(figsize=(8,5))

plt.hist(
    important_values,
    bins=bins,
    alpha=0.5,
    edgecolor="black",
    label="Kernel (0.1)"
)

plt.hist(
    important_values1,
    bins=bins,
    alpha=0.5,
    edgecolor="black",
    label="LRP"
)

plt.xlabel("Pupil Diameter (mm)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
bins = np.histogram_bin_edges(
    np.concatenate([important_values, important_values1]),
    bins=50
)

counts1, _ = np.histogram(important_values, bins=bins)
counts2, _ = np.histogram(important_values1, bins=bins)

ymax = max(counts1.max() + 1000, counts2.max()+ 1000)

fig, axes = plt.subplots(2, 1, figsize=(8,8), sharex=True)

axes[0].hist(important_values, bins=bins, edgecolor="black")
axes[0].set_ylim(0, ymax)
axes[0].set_title("Kernel (0.1)")

axes[1].hist(important_values1, bins=bins, edgecolor="black")
axes[1].set_ylim(0, ymax)
axes[1].set_title("LRP")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 8), sharex=True)

# ----- Top 5% -----
axes[0].plot(signal, color="steelblue", lw=1.5)
axes[0].scatter(
    top5,
    signal[top5],
    color="red",
    s=20,
    label="Top 5%"
)
axes[0].set_title("ROCKET-LRP Top 5% Relevance")
axes[0].legend()

# ----- Top 10% -----
axes[1].plot(signal, color="steelblue", lw=1.5)
axes[1].scatter(
    top10,
    signal[top10],
    color="darkorange",
    s=20,
    label="Top 10%"
)
axes[1].set_title("ROCKET-LRP Top 10% Relevance")
axes[1].legend()

# ----- Top 15% -----
axes[2].plot(signal, color="steelblue", lw=1.5)
axes[2].scatter(
    top15,
    signal[top15],
    color="green",
    s=20,
    label="Top 15%"
)
axes[2].set_title("ROCKET-LRP Top 15% Relevance")
axes[2].set_xlabel("Time")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

n_timepoints = X_test.shape[2]

# Convert to sets for fast lookup
kernel_set = set(important_indices)
lrp_set = set(important_indices1)

both_flagged = 0
kernel_only = 0
lrp_only = 0
none_flagged = 0

for t in range(n_timepoints):
    kernel = t in kernel_set
    lrp = t in lrp_set

    if kernel and lrp:
        both_flagged += 1
    elif kernel and not lrp:
        kernel_only += 1
    elif not kernel and lrp:
        lrp_only += 1
    else:
        none_flagged += 1

table = pd.DataFrame(
    [[both_flagged, kernel_only],
     [lrp_only, none_flagged]],
    index=["LRP Flagged", "LRP Not Flagged"],
    columns=["Kernel Flagged", "Kernel Not Flagged"]
)

print(table)

In [ ]:
important_indices = important_indices.reshape(X_test.shape[0], k)
important_indices1 = important_indices1.reshape(X_test.shape[0], k)

In [ ]:
important_indices[0]

In [ ]:
common = 0
lrp = 0
fdf = 0
neither = 0 
n_timepoints = X_test.shape[2]

for i in range(X_test.shape[0]):
    fdf_set = set(important_indices[i])
    lrp_set = set(important_indices1[i])
    common += len(set(important_indices[i]) & set(important_indices1[i]))
    lrp += len(set(important_indices1[i]) - set(important_indices[i]))
    fdf += len(set(important_indices[i]) - set(important_indices1[i]))
    neither += n_timepoints - len(fdf_set | lrp_set)

In [ ]:
import pandas as pd

table = pd.DataFrame(
    [[common, fdf],
     [lrp, neither]],
    index=["LRP Flagged", "LRP Not Flagged"],
    columns=["FDF Flagged", "FDF Not Flagged"]
)

# Add row totals
#table["Total"] = table.sum(axis=1)

# Add column totals (including the grand total)
#table.loc["Total"] = table.sum(axis=0)

print(table)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sample = 6

signal = X_test[sample, 0]

# -----------------------------
# LRP top 5%
# -----------------------------
importance_lrp = np.abs(R_lrp[sample, 0])

k = int(0.05 * len(signal))

top5_lrp = np.argsort(importance_lrp)[-k:]

# -----------------------------
# Kernel 0.1 top 5%
# -----------------------------
top5_kernel = np.argsort(importance01)[-k:]


# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(16, 4))

# Raw signal
ax.plot(signal, color="black", linewidth=1, label="Signal")


# Get y limits for full-height rectangles
ymin, ymax = ax.get_ylim()

# Rectangle width (half of normal point width)
width = 0.5


# -----------------------------
# LRP rectangles
# -----------------------------
for idx in top5_lrp:
    ax.axvspan(
        idx - width/2,
        idx + width/2,
        color="red",
        alpha=0.35,
        label="ROCKET-LRP Top 5%" if idx == top5_lrp[0] else None
    )


# -----------------------------
# Kernel rectangles
# -----------------------------
for idx in top5_kernel:
    ax.axvspan(
        idx - width/2,
        idx + width/2,
        color="yellow",
        alpha=0.35,
        label="FDF-ROCKET (0.1) Top 5%" if idx == top5_kernel[0] else None
    )


ax.set_xlabel("Time")
ax.set_ylabel("Signal")
ax.set_title(f"Sample {sample}: Top 5% Important Time Points")

ax.legend()
plt.tight_layout()
plt.show()